# Credit Card Fraud Detection

This notebook demonstrates how to process the Kaggle Credit Card Fraud dataset, handle class imbalance using SMOTE, and train several machine learning models to identify fraudulent transactions.

## Objectives:
1. Data loading and preprocessing (normalize Amount/Time to handle varying scales, train-test split).
2. Address the severe class imbalance with Extreme Minority Oversampling (SMOTE) technique.
3. Train and compare 4 distinct model types:
   - Logistic Regression
   - Random Forest Classifier
   - XGBoost Classifier
   - Isolation Forest (Unsupervised Anomaly Detection)
4. Evaluate performance using Precision, Recall, F1-Score, and ROC-AUC.
5. Plot confusion matrices and precision-recall curves to visualize false positives and false negatives clearly.

In [ ]:
# Imports block
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
import xgboost as xgb

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, precision_recall_curve, auc, classification_report

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading
Assuming the `creditcard.csv` file from Kaggle is located in the same directory.

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('creditcard.csv')
    print("Dataset loaded successfully.")
    print(f"Shape: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print("Error: 'creditcard.csv' not found. Please download the dataset from Kaggle and place it in the same directory.")
    # Creating dummy data strictly for the notebook to run out of the box if no data provided
    print("Creating dummy data for demonstration...")
    df = pd.DataFrame(np.random.rand(1000, 30), columns=['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount'])
    df['Class'] = np.random.choice([0, 1], size=1000, p=[0.99, 0.01])

## 2. Data Preprocessing
We need to normalize the `Amount` and `Time` features since the `V1`-`V28` features (result of PCA) are already scaled.
Then, we split the data into train and test sets.

In [ ]:
# Normalize 'Amount' and 'Time'
scaler = StandardScaler()
df['Scaled_Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['Scaled_Time'] = scaler.fit_transform(df['Time'].values.reshape(-1, 1))

# Drop original unscaled features
df.drop(['Amount', 'Time'], axis=1, inplace=True)

# Reorder columns for convenience
scaled_amount = df['Scaled_Amount']
scaled_time = df['Scaled_Time']
df.drop(['Scaled_Amount', 'Scaled_Time'], axis=1, inplace=True)
df.insert(0, 'Scaled_Amount', scaled_amount)
df.insert(1, 'Scaled_Time', scaled_time)

# Separate Input features (X) and Target variable (y)
X = df.drop('Class', axis=1)
y = df['Class']

# Train-test split (80-20 split)
# Applying stratify=y is critical here to ensure the 80/20 train/test split maintains the extreme class imbalance ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training Set Shape:", X_train.shape, y_train.shape)
print("Testing Set Shape:", X_test.shape, y_test.shape)
print("Fraud transactions in training set:", sum(y_train == 1))
print("Fraud transactions in test set:", sum(y_test == 1))

## 3. Handle Class Imbalance with SMOTE
SMOTE (Synthetic Minority Over-sampling Technique) creates synthetic data points for the minority class to balance the target feature and prevent model bias.
**Important:** We ONLY apply SMOTE on the training dataset to ensure no data leakage into the test set.

In [ ]:
smote = SMOTE(random_state=42)

# Resample the training dataset
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE Oversampling:")
print("Training Set Shape:", X_train_smote.shape)
print("Target values:\n", y_train_smote.value_counts())

## 4. Model Training and Evaluation Helper Function
Since we're testing multiple models, a reusable evaluation function reduces clutter.

In [ ]:
results_metrics = {}

def evaluate_model(model_name, y_true, y_pred, y_prob):
    print(f"========== Evaluation for {model_name} ==========")
    
    # Calculate core metrics
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob)
    
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    print("\nClassification Report:\n", classification_report(y_true, y_pred, zero_division=0))
    
    # 1. Plot Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(14, 5))
    
    plt.subplot(1, 2, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, 
                xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'])
    plt.title(f'Confusion Matrix: {model_name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    
    # 2. Plot Precision-Recall Curve
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall_curve, precision_curve)
    
    plt.subplot(1, 2, 2)
    plt.plot(recall_curve, precision_curve, marker='.', label=f'PR-AUC = {pr_auc:.4f}')
    plt.title(f'Precision-Recall Curve: {model_name}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return {'Precision': precision, 'Recall': recall, 'F1-Score': f1, 'ROC-AUC': roc_auc, 'PR-AUC': pr_auc}

### 4.1 Logistic Regression

In [ ]:
# Initialize and train
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_smote, y_train_smote)

# Predict on pure test set
y_pred_lr = lr_model.predict(X_test)
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

results_metrics['Logistic Regression'] = evaluate_model('Logistic Regression', y_test, y_pred_lr, y_prob_lr)

### 4.2 Random Forest Classifier

In [ ]:
# Initialize and train (Limited depth & estimators to keep notebook execution reasonably fast)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train_smote, y_train_smote)

# Predict
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

results_metrics['Random Forest'] = evaluate_model('Random Forest', y_test, y_pred_rf, y_prob_rf)

### 4.3 XGBoost Classifier

In [ ]:
# Initialize and train
xgb_model = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', n_jobs=-1)
xgb_model.fit(X_train_smote, y_train_smote)

# Predict
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

results_metrics['XGBoost'] = evaluate_model('XGBoost', y_test, y_pred_xgb, y_prob_xgb)

### 4.4 Isolation Forest (Unsupervised Approach)
Isolation forest works differently. It models 'normal' data and flags anomalies. We train it strictly on the non-SMOTE, original training features as it behaves poorly with synthetically oversampled data. It natively seeks out extremely rare occurrences in multidimensional space.

In [ ]:
# Initialize and fit ONLY ON X_train (not smote!)
# Contamination is the expected proportion of outliers (roughly ~0.17% in full dataset)
contamination_est = 0.0017 
iso_model = IsolationForest(n_estimators=100, max_samples=len(X_train), contamination=contamination_est, random_state=42, n_jobs=-1)

# We fit on X_train. It is completely unsupervised with respect to class labels.
iso_model.fit(X_train)

# Predict (Returns 1 for normal, -1 for anomaly)
y_pred_iso_raw = iso_model.predict(X_test)

# Map to 0 (normal) and 1 (anomaly) to match our dataset ground truth formats
y_pred_iso = np.where(y_pred_iso_raw == -1, 1, 0)

# Get anomaly score (the lower, the more abnormal). 
# We flip the sign so higher score = higher probability of fraud.
y_prob_iso = -iso_model.score_samples(X_test)

results_metrics['Isolation Forest'] = evaluate_model('Isolation Forest', y_test, y_pred_iso, y_prob_iso)

## 5. Model Comparison Summary

In [ ]:
# Structure all results into a dataframe and sort by Area Under Precision-Recall Curve (PR-AUC)
# PR-AUC is usually the best metric for extreme class imbalance contexts
summary_df = pd.DataFrame(results_metrics).T
display(summary_df.sort_values(by='PR-AUC', ascending=False))

# Optional: Visualize comparison
summary_df[['F1-Score', 'ROC-AUC', 'PR-AUC']].plot(kind='bar', figsize=(10, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim([0, 1.0])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()